# Timeline: Multi-Frame Dynamic Scene Processing

Demonstrates the `Timeline` class for generating multi-frame radar sequences from moving scenes.

**Pipeline:**
1. Ray-trace a static mesh scene to get reflection points
2. Create keyframes by shifting points to simulate motion
3. `Timeline` interpolates between keyframes at continuous time
4. `timeline.generate()` produces full MIMO frames for each radar frame
5. `timeline.generate_rd()` produces Range-Doppler maps per frame

**Requires:** mitsuba (cuda_ad_rgb variant)

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[0]))

import torch
import numpy as np
import matplotlib.pyplot as plt
from witwin.radar import Radar, Renderer, Scene, Timeline
from witwin.core import Box


## 1. Setup: Radar + Scene + Ray Trace

Same mesh scene as `mesh_scene.ipynb` — wall + two boxes.

In [ ]:
config = {
    "num_tx": 3, "num_rx": 4,
    "fc": 77e9,
    "slope": 60.012,
    "adc_samples": 256,
    "adc_start_time": 6,
    "sample_rate": 4400,
    "idle_time": 7,
    "ramp_end_time": 65,
    "chirp_per_frame": 128,
    "frame_per_second": 10,
    "num_doppler_bins": 128,
    "num_range_bins": 256,
    "num_angle_bins": 64,
    "power": 15,
    "tx_loc": [[0, 0, 0], [4, 0, 0], [2, 1, 0]],
    "rx_loc": [[-6, 0, 0], [-5, 0, 0], [-4, 0, 0], [-3, 0, 0]],
}
radar = Radar(config)

scene = Scene()
scene.set_sensor(origin=(0, 0, 0), target=(0, 0, -5), fov=60)

wall_v, wall_f = Box(position=(0, 0, -5), size=(6, 4, 0.01)).to_mesh()
scene.add_mesh(name="wall", vertices=wall_v, faces=wall_f)
box_v, box_f = Box(position=(0.5, 0, -3), size=(0.8, 0.8, 0.8)).to_mesh()
scene.add_mesh(name="box_a", vertices=box_v, faces=box_f)
box2_v, box2_f = Box(position=(-1.0, -0.5, -4), size=(0.6, 1.0, 0.6)).to_mesh()
scene.add_mesh(name="box_b", vertices=box2_v, faces=box2_f)

renderer = Renderer(scene, resolution=128)
points, intensities = renderer.trace()
print(f"{points.shape[0]} reflection points")


## 2. Build Timeline

Simulate 3 keyframes at 30 FPS — scene moves 0.5 m/s toward radar along Z.

In [ ]:
velocity = torch.tensor([0.0, 0.0, -0.5], device='cuda')  # 0.5 m/s toward radar
dt = 1.0 / 30  # source frame interval

timeline = Timeline(frame_rate=30)

for i in range(3):
    offset = velocity * (i * dt)
    timeline.add_keyframe(points + offset, intensities)

print(f"Keyframes: {timeline.num_frames}")
print(f"Duration:  {timeline.duration:.4f} s")
print(f"Expected radar frames: {max(1, int(timeline.duration * radar.frame_per_second))}")

## 3. Interpolator

The interpolator is a closure `f(t) → (intensities, positions)` that linearly blends between adjacent keyframes.

In [ ]:
interp = timeline.get_interpolator()

# Query at t=0 (first keyframe)
ints_0, pos_0 = interp(0.0)
print(f"t=0.000s: {pos_0.shape[0]} points")

# Query at midpoint
t_mid = timeline.duration / 2
ints_m, pos_m = interp(t_mid)
print(f"t={t_mid:.4f}s: {pos_m.shape[0]} points")

# Query at end
ints_e, pos_e = interp(timeline.duration)
print(f"t={timeline.duration:.4f}s: {pos_e.shape[0]} points")

## 4. Generate Radar Frames

In [ ]:
frames = timeline.generate(radar, progress=True)
print(f"Output shape: {frames.shape}  (num_frames, TX, RX, chirps, ADC)")

## 5. Range-Doppler Maps

In [ ]:
rd_mags, rd_ranges, rd_velocities = timeline.generate_rd(radar, tx=0, rx=0, progress=False)
print(f"RD maps shape: {rd_mags.shape}  (num_frames, chirps, ADC)")

n_rd = rd_mags.shape[0]
fig, axes = plt.subplots(1, n_rd, figsize=(6 * n_rd, 4))
if n_rd == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.imshow(
        rd_mags[i, :, :len(rd_ranges)],
        extent=[rd_ranges[0], rd_ranges[-1], rd_velocities[0], rd_velocities[-1]],
        aspect='auto', origin='lower', cmap='jet',
    )
    ax.set_xlabel('Range (m)')
    ax.set_ylabel('Velocity (m/s)')
    ax.set_title(f'Frame {i}')

fig.suptitle('Timeline: Range-Doppler per radar frame')
plt.tight_layout()

## 6. Bulk Add via `add_pointcloud_sequence`

Alternative API — pass a list of point clouds at once.

In [ ]:
pc_list = [points + velocity * (i * dt) for i in range(5)]

tl2 = Timeline(frame_rate=30)
tl2.add_pointcloud_sequence(pc_list)

print(f"Keyframes: {tl2.num_frames}")
print(f"Duration:  {tl2.duration:.4f} s")